In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed/retail_inventory_cleaned.csv")

df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Units Sold"] = pd.to_numeric(df["Units Sold"], errors="coerce")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (73100, 14)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Year', 'Month', 'DayOfWeek', 'Revenue']


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Year,Month,DayOfWeek,Revenue
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,2022,1,5,4254.50
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,2022,1,5,9451.50
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,2022,1,5,1819.35
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,2022,1,5,1995.92
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,2022,1,5,1030.96


In [2]:
treatment = "Price"
outcome = "Units Sold"

candidate_controls = [
    "Product ID",
    "Store ID",
    "Category",
    "Region",
    "Inventory Level",
    "Date"
]

print("Treatment:", treatment)
print("Outcome:", outcome)
print("Candidate controls:", candidate_controls)

Treatment: Price
Outcome: Units Sold
Candidate controls: ['Product ID', 'Store ID', 'Category', 'Region', 'Inventory Level', 'Date']


In [3]:
required_columns = [
    "Date",
    "Price",
    "Units Sold",
    "Product ID",
    "Store ID",
    "Category",
    "Region"
]

print("Missing values in required columns:")
display(df[required_columns].isna().sum())

analysis_df = df.dropna(subset=required_columns).copy()
analysis_df = analysis_df[analysis_df["Price"] > 0].copy()

print("\nOriginal rows:", len(df))
print("Rows retained:", len(analysis_df))
print("Rows removed:", len(df) - len(analysis_df))
print("Date range:",
      analysis_df["Date"].min(),
      "to",
      analysis_df["Date"].max())

Missing values in required columns:


Date          0
Price         0
Units Sold    0
Product ID    0
Store ID      0
Category      0
Region        0
dtype: int64


Original rows: 73100
Rows retained: 73100
Rows removed: 0
Date range: 2022-01-01 00:00:00 to 2024-01-01 00:00:00


In [4]:
pair_summary = analysis_df.groupby(
    ["Product ID", "Store ID"]
).agg(
    observations=("Price", "size"),
    unique_prices=("Price", "nunique"),
    minimum_price=("Price", "min"),
    maximum_price=("Price", "max")
)

pair_summary["price_range"] = (
    pair_summary["maximum_price"]
    - pair_summary["minimum_price"]
)

print("Total product-store pairs:", len(pair_summary))
print(
    "Pairs with changing prices:",
    (pair_summary["unique_prices"] > 1).sum()
)
print(
    "Pairs with constant prices:",
    (pair_summary["unique_prices"] == 1).sum()
)

display(pair_summary.head(10))

Total product-store pairs: 100
Pairs with changing prices: 100
Pairs with constant prices: 0


observations  unique_prices  minimum_price  \
Product ID Store ID                                               
P0001      S001               731            697          10.26   
           S002               731            694          10.81   
           S003               731            693          10.19   
           S004               731            694          10.29   
           S005               731            709          10.07   
P0002      S001               731            697          10.15   
           S002               731            706          10.01   
           S003               731            715          10.16   
           S004               731            706          10.60   
           S005               731            705          10.08   

                     maximum_price  price_range  
Product ID Store ID                              
P0001      S001              99.97        89.71  
           S002              99.99        89.18  
           S003              99.86        89.67  
           S004              99.88        89.59  
           S005              99.76        89.69  
P0002      S001              99.95        89.80  
           S002              99.99        89.98  
           S003              99.67        89.51  
           S004              99.96        89.36  
           S005              99.99        89.91

In [5]:
analysis_df["Year"] = analysis_df["Date"].dt.year
analysis_df["Month"] = analysis_df["Date"].dt.month
analysis_df["DayOfWeek"] = analysis_df["Date"].dt.dayofweek

print("Unique dates:", analysis_df["Date"].nunique())
print("Unique categories:", analysis_df["Category"].nunique())
print("Unique regions:", analysis_df["Region"].nunique())

display(
    analysis_df[
        [
            "Price",
            "Units Sold",
            "Inventory Level",
            "Category",
            "Region",
            "Year",
            "Month",
            "DayOfWeek"
        ]
    ].describe(include="all").T
)

Unique dates: 731
Unique categories: 5
Unique regions: 4


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Price,73100.0,NaN,NaN,NaN,55.135108,26.021945,10.0,32.65,55.05,77.86,100.0
Units Sold,73100.0,NaN,NaN,NaN,136.46487,108.919406,0.0,49.0,107.0,203.0,499.0
Inventory Level,73100.0,NaN,NaN,NaN,274.469877,129.949514,50.0,162.0,273.0,387.0,500.0
Category,73100,5,Furniture,14699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,73100,4,East,18349,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,73100.0,NaN,NaN,NaN,2022.502052,0.502728,2022.0,2022.0,2023.0,2023.0,2024.0
Month,73100.0,NaN,NaN,NaN,6.518468,3.451564,1.0,4.0,7.0,10.0,12.0
DayOfWeek,73100.0,NaN,NaN,NaN,3.002736,2.003429,0.0,1.0,3.0,5.0,6.0


In [6]:
output_path = Path(
    "../data/processed/dml_preparation.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Prepared dataset shape:", analysis_df.shape)

Saved to: ../data/processed/dml_preparation.csv
Prepared dataset shape: (73100, 14)
